# 04 Model Training for Women's Health Prediction

This notebook trains multiple models to predict mood, energy, and stress levels based on cycle data.

## Models to Train:
- Random Forest Regressor
- Gradient Boosting (XGBoost)
- LSTM Neural Network
- Ensemble Model

## Prediction Targets:
- Mood (-2 to +2)
- Energy (-2 to +2) 
- Stress Level (-3 to 0)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import joblib
from pathlib import Path

# ML Libraries
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.multioutput import MultiOutputRegressor

# Deep Learning
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# XGBoost
import xgboost as xgb

import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print("Libraries imported successfully!")
print(f"TensorFlow version: {tf.__version__}")
print(f"XGBoost version: {xgb.__version__}")

## 1. Load and Prepare Data

In [ ]:
# Load feature-engineered data
df = pd.read_csv('data/processed/daily_data_with_features.csv')
print(f"Loaded data shape: {df.shape}")

# Load feature names
with open('data/processed/feature_names.json', 'r') as f:
    feature_names = json.load(f)

print(f"\nTarget variables: {feature_names['target_variables']}")
print(f"Number of numerical features: {len(feature_names['numerical_features'])}")
print(f"Number of categorical features: {len(feature_names['categorical_features'])}")

# Check data quality
print(f"\nData quality:")
print(f"Missing values: {df.isnull().sum().sum():,}")
print(f"Duplicate rows: {df.duplicated().sum():,}")

# Display basic statistics for targets
target_cols = feature_names['target_variables']
print(f"\nTarget variable statistics:")
display(df[target_cols].describe())

In [ ]:
# Prepare features and targets
def prepare_data_for_training(df, feature_names):
    """Prepare data for model training"""
    
    # Define features to use (exclude non-predictive columns)
    exclude_cols = ['user_id', 'ClientID', 'original_index', 'base_user_id', 'symptoms']
    
    # Get all potential feature columns
    all_feature_cols = [col for col in df.columns 
                       if col not in target_cols + exclude_cols]
    
    # Remove features with too many missing values (>50%)
    missing_pct = df[all_feature_cols].isnull().mean()
    valid_features = missing_pct[missing_pct < 0.5].index.tolist()
    
    print(f"Features after removing high-missing columns: {len(valid_features)}")
    
    # Prepare feature matrix
    X = df[valid_features].copy()
    
    # Handle remaining missing values
    # For numerical features: fill with median
    numerical_cols = X.select_dtypes(include=[np.number]).columns
    X[numerical_cols] = X[numerical_cols].fillna(X[numerical_cols].median())
    
    # For categorical features: fill with mode or 'unknown'
    categorical_cols = X.select_dtypes(exclude=[np.number]).columns
    for col in categorical_cols:
        X[col] = X[col].fillna(X[col].mode()[0] if not X[col].mode().empty else 'unknown')
    
    # Encode categorical features
    label_encoders = {}
    for col in categorical_cols:
        le = LabelEncoder()
        X[col] = le.fit_transform(X[col].astype(str))
        label_encoders[col] = le
    
    # Prepare target matrix
    y = df[target_cols].copy()
    
    # Remove rows where all targets are missing
    valid_rows = ~y.isnull().all(axis=1)
    X = X[valid_rows]
    y = y[valid_rows]
    
    print(f"Final dataset shape: X={X.shape}, y={y.shape}")
    print(f"Missing values in targets: {y.isnull().sum().sum()}")
    
    return X, y, valid_features, label_encoders

X, y, feature_cols, encoders = prepare_data_for_training(df, feature_names)

# Display feature importance preview
print(f"\nFeature columns ({len(feature_cols)}):")
for i, col in enumerate(feature_cols[:15]):
    print(f"{i+1:2d}. {col}")
if len(feature_cols) > 15:
    print(f"    ... and {len(feature_cols) - 15} more features")

## 2. Data Splitting and Scaling

In [ ]:
# Split data into train/validation/test
# Use stratified split based on user_id to ensure users don't leak between sets
user_ids = df.loc[X.index, 'user_id'].str.split('_').str[:-1].str.join('_')
unique_users = user_ids.unique()

# Split users into train/val/test
train_users, temp_users = train_test_split(unique_users, test_size=0.4, random_state=42)
val_users, test_users = train_test_split(temp_users, test_size=0.5, random_state=42)

print(f"User splits:")
print(f"  Train users: {len(train_users)}")
print(f"  Validation users: {len(val_users)}")
print(f"  Test users: {len(test_users)}")

# Create data splits
train_mask = user_ids.isin(train_users)
val_mask = user_ids.isin(val_users)
test_mask = user_ids.isin(test_users)

X_train, y_train = X[train_mask], y[train_mask]
X_val, y_val = X[val_mask], y[val_mask]
X_test, y_test = X[test_mask], y[test_mask]

print(f"\nData splits:")
print(f"  Train: {X_train.shape[0]:,} samples ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"  Validation: {X_val.shape[0]:,} samples ({X_val.shape[0]/len(X)*100:.1f}%)")
print(f"  Test: {X_test.shape[0]:,} samples ({X_test.shape[0]/len(X)*100:.1f}%)")

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print(f"\nData scaling completed")
print(f"Feature means: {X_train_scaled.mean(axis=0)[:5]}")
print(f"Feature stds: {X_train_scaled.std(axis=0)[:5]}")

## 3. Baseline Models

In [ ]:
# Define evaluation metrics
def evaluate_model(y_true, y_pred, model_name="Model"):
    """Evaluate model performance"""
    results = {}
    
    for i, target in enumerate(target_cols):
        y_true_target = y_true.iloc[:, i].dropna()
        y_pred_target = y_pred[:, i][y_true.iloc[:, i].notna()]
        
        if len(y_true_target) > 0:
            mse = mean_squared_error(y_true_target, y_pred_target)
            mae = mean_absolute_error(y_true_target, y_pred_target)
            r2 = r2_score(y_true_target, y_pred_target)
            
            results[target] = {
                'MSE': mse,
                'RMSE': np.sqrt(mse),
                'MAE': mae,
                'R2': r2
            }
    
    return results

# Train baseline models
models = {}
model_results = {}

print("Training baseline models...\n")

# 1. Linear Regression
print("1. Linear Regression")
lr_model = MultiOutputRegressor(Ridge(alpha=1.0))
lr_model.fit(X_train_scaled, y_train)
y_pred_lr = lr_model.predict(X_val_scaled)
lr_results = evaluate_model(y_val, y_pred_lr, "Linear Regression")
models['Linear Regression'] = lr_model
model_results['Linear Regression'] = lr_results

for target, metrics in lr_results.items():
    print(f"  {target}: RMSE={metrics['RMSE']:.3f}, MAE={metrics['MAE']:.3f}, R2={metrics['R2']:.3f}")

# 2. Random Forest
print("\n2. Random Forest")
rf_model = MultiOutputRegressor(RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1))
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_val)
rf_results = evaluate_model(y_val, y_pred_rf, "Random Forest")
models['Random Forest'] = rf_model
model_results['Random Forest'] = rf_results

for target, metrics in rf_results.items():
    print(f"  {target}: RMSE={metrics['RMSE']:.3f}, MAE={metrics['MAE']:.3f}, R2={metrics['R2']:.3f}")

print("\nBaseline models trained!")

## 4. Model Comparison and Visualization

In [ ]:
# Compare all models
def create_comparison_table(model_results):
    """Create comparison table for all models"""
    comparison_data = []
    
    for model_name, results in model_results.items():
        for target, metrics in results.items():
            comparison_data.append({
                'Model': model_name,
                'Target': target,
                'RMSE': metrics['RMSE'],
                'MAE': metrics['MAE'],
                'R2': metrics['R2']
            })
    
    return pd.DataFrame(comparison_data)

comparison_df = create_comparison_table(model_results)
print("Model Comparison:")
display(comparison_df.pivot_table(index='Model', columns='Target', values=['RMSE', 'MAE', 'R2']))

# Find best model for each target
best_models = {}
for target in target_cols:
    target_results = comparison_df[comparison_df['Target'] == target]
    best_model_idx = target_results['R2'].idxmax()
    best_model_name = target_results.loc[best_model_idx, 'Model']
    best_models[target] = best_model_name
    print(f"Best model for {target}: {best_model_name} (R2={target_results.loc[best_model_idx, 'R2']:.3f})")

# Visualize model performance
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# RMSE comparison
rmse_pivot = comparison_df.pivot_table(index='Model', columns='Target', values='RMSE')
rmse_pivot.plot(kind='bar', ax=axes[0,0])
axes[0,0].set_title('RMSE by Model and Target')
axes[0,0].set_ylabel('RMSE')
axes[0,0].tick_params(axis='x', rotation=45)

# R2 comparison
r2_pivot = comparison_df.pivot_table(index='Model', columns='Target', values='R2')
r2_pivot.plot(kind='bar', ax=axes[0,1])
axes[0,1].set_title('R² by Model and Target')
axes[0,1].set_ylabel('R² Score')
axes[0,1].tick_params(axis='x', rotation=45)

# Feature importance from Random Forest
if 'Random Forest' in models:
    feature_importance = np.mean([
        estimator.feature_importances_ 
        for estimator in models['Random Forest'].estimators_
    ], axis=0)
    
    top_features_idx = np.argsort(feature_importance)[-15:]
    top_features = np.array(feature_cols)[top_features_idx]
    top_importance = feature_importance[top_features_idx]
    
    axes[1,0].barh(range(len(top_features)), top_importance)
    axes[1,0].set_yticks(range(len(top_features)))
    axes[1,0].set_yticklabels([f.split('_')[0] + '_' + f.split('_')[1] if len(f.split('_')) > 1 else f for f in top_features])
    axes[1,0].set_title('Top 15 Feature Importance')
    axes[1,0].set_xlabel('Importance')

# Prediction scatter plot for best model
best_predictions = models['Random Forest'].predict(X_val)
axes[1,1].scatter(y_val.iloc[:, 0], best_predictions[:, 0], alpha=0.5, label=target_cols[0])
axes[1,1].plot([y_val.iloc[:, 0].min(), y_val.iloc[:, 0].max()], 
               [y_val.iloc[:, 0].min(), y_val.iloc[:, 0].max()], 'r--', lw=2)
axes[1,1].set_xlabel('Actual')
axes[1,1].set_ylabel('Predicted')
axes[1,1].set_title('Actual vs Predicted')

plt.tight_layout()
plt.show()

## 5. Final Model Evaluation on Test Set

In [ ]:
# Evaluate best models on test set
print("Evaluating models on test set...\n")

test_results = {}

# Test Random Forest
y_test_pred_rf = models['Random Forest'].predict(X_test)
rf_test_results = evaluate_model(y_test, y_test_pred_rf, "Random Forest Test")
test_results['Random Forest'] = rf_test_results

print("Random Forest Test Results:")
for target, metrics in rf_test_results.items():
    print(f"  {target}: RMSE={metrics['RMSE']:.3f}, MAE={metrics['MAE']:.3f}, R2={metrics['R2']:.3f}")

# Create final comparison
final_comparison = create_comparison_table(test_results)
print("\n" + "="*60)
print("FINAL TEST SET COMPARISON:")
print("="*60)
display(final_comparison.pivot_table(index='Model', columns='Target', values=['RMSE', 'MAE', 'R2']))

## 6. Save Models and Results

In [ ]:
# Create models directory
models_dir = Path('models/trained')
models_dir.mkdir(parents=True, exist_ok=True)

print("Saving models and artifacts...\n")

# Save preprocessing objects
preprocessing_artifacts = {
    'scaler': scaler,
    'label_encoders': encoders,
    'feature_columns': feature_cols,
    'target_columns': target_cols
}

joblib.dump(preprocessing_artifacts, models_dir / 'preprocessing_artifacts.joblib')
print("Preprocessing artifacts saved")

# Save best traditional models
joblib.dump(models['Random Forest'], models_dir / 'random_forest_model.joblib')
print("Random Forest model saved")

# Save model results
results_summary = {
    'validation_results': model_results,
    'test_results': test_results,
    'best_models': best_models,
    'training_info': {
        'train_samples': len(X_train),
        'val_samples': len(X_val),
        'test_samples': len(X_test),
        'n_features': len(feature_cols),
        'n_targets': len(target_cols)
    }
}

with open(models_dir / 'model_results.json', 'w') as f:
    json.dump(results_summary, f, indent=2, default=str)
print("Model results saved")

# Final summary
print(f"\n" + "="*60)
print("MODEL TRAINING COMPLETED SUCCESSFULLY!")
print("="*60)
print(f"Models saved in: {models_dir}")
print(f"Best performing models:")
for target, model_name in best_models.items():
    if target in test_results[model_name]:
        r2 = test_results[model_name][target]['R2']
        print(f"   {target}: {model_name} (R² = {r2:.3f})")

print(f"\nTotal features used: {len(feature_cols)}")
print(f"Training samples: {len(X_train):,}")
print(f"\nReady for deployment and further analysis!")